# This file generates the cleaned data frame we will use for other anlayses 
Load in data, merge demographics and ABC scores, map demographics, generate preliminary tables based on demographic counts

In [1]:
import numpy as np
import pandas as pd

In [2]:
# normalize ABC score on a scale to 1-100
def normalize_to_100(value, max_value):
    return (value / max_value) * 100

In [3]:
# read in demographics file 
demographics = pd.read_csv("../data/Dartmouth Data Set- SIRS .xlsx - Demographics.csv")
print(demographics.head())
print(demographics.shape)
print(demographics['Local ID'].nunique())

# get local IDs
local_ids = demographics['Local ID']

   Local ID Date Enrolled in START  Status Status Date  \
0  11128335               7/1/2010  Active   7/22/2010   
1  11134104              11/1/2011  Active  11/30/2011   
2  11138520               5/1/2012  Active   5/21/2012   
3  11132094               5/1/2013  Active    5/9/2013   
4   975134C               7/1/2014  Active    7/1/2014   

                       Time Enrolled in START  \
0  Business Hours (Monday - Friday 8am - 5pm)   
1  Business Hours (Monday - Friday 8am - 5pm)   
2  Business Hours (Monday - Friday 8am - 5pm)   
3  Business Hours (Monday - Friday 8am - 5pm)   
4  Business Hours (Monday - Friday 8am - 5pm)   

        Source of referral to START Suitability of enrollment in START  \
0  Case Manager/Service Coordinator                        Appropriate   
1  Case Manager/Service Coordinator                        Appropriate   
2  Case Manager/Service Coordinator                        Appropriate   
3  Case Manager/Service Coordinator                        A

In [4]:
# read in ABC score file 
abc = pd.read_csv("../data/Dartmouth Data Set- SIRS .xlsx - ABC Data.csv")
print(abc.head())
print(abc.shape)
print(abc['Local ID'].nunique())

# filter out entries not represented in demographic data 
abc_filtered = abc[abc['Local ID'].isin(local_ids)]
print(abc_filtered['Local ID'].nunique())

   Local ID Date Reviewed Type of Service  Irritability/Agitation  \
0  11163106      2/1/2013          Intake                     9.0   
1  11163106      2/1/2013          Intake                     9.0   
2  11134102      3/1/2013          Intake                    21.0   
3  11134102      3/1/2013          Intake                    21.0   
4  11134102     11/1/2014   Re-evaluation                    15.0   

   Lethargy/Social Withdrawal  Stereotypic Behavior  \
0                        16.0                   0.0   
1                        16.0                   0.0   
2                        21.0                   2.0   
3                        21.0                   2.0   
4                        11.0                   2.0   

   Hyperactivity/Noncompliance  Inappropriate Speech  
0                          0.0                   0.0  
1                          0.0                   0.0  
2                          9.0                   2.0  
3                          9.0    

In [5]:
# merge ABC and demographic files based on local ID 

print("abc rows: " + str(len(abc_filtered)))
print("demographic rows:" + str(len(demographics)))
demographic_abcs = pd.merge(abc_filtered,
                  demographics, on="Local ID")

demographic_abcs
print("merged rows:" + str(len(demographic_abcs)))

abc rows: 19461
demographic rows:4986
merged rows:19461


In [6]:
# normalize ABC scores to be on a scale from 0-100 

demographic_abcs['norm_leth_soc'] = demographic_abcs['Lethargy/Social Withdrawal'].apply(lambda x: normalize_to_100(x, 48))
demographic_abcs['norm_ir_agit'] = demographic_abcs['Irritability/Agitation'].apply(lambda x: normalize_to_100(x, 45))
demographic_abcs['norm_stereo'] = demographic_abcs['Stereotypic Behavior'].apply(lambda x: normalize_to_100(x, 21))
demographic_abcs['norm_hyper'] = demographic_abcs['Hyperactivity/Noncompliance'].apply(lambda x: normalize_to_100(x, 48))
demographic_abcs['norm_speech'] = demographic_abcs['Inappropriate Speech'].apply(lambda x: normalize_to_100(x, 12))

demographic_abcs.head()

,Local ID,Date Reviewed,Type of Service,Irritability/Agitation,Lethargy/Social Withdrawal,Stereotypic Behavior,Hyperactivity/Noncompliance,Inappropriate Speech,Date Enrolled in START,Status,...,Current IEP,Current 504 Plan,School Setting,Custody Status,Child Lives With,norm_leth_soc,norm_ir_agit,norm_stereo,norm_hyper,norm_speech
0,11153125,10/26/2016,Re-evaluation,3.0,2.0,0.0,0.0,0.0,4/1/2013,Active,...,No,No,NaN,NaN,NaN,4.166667,6.666667,0.000000,0.000000,0.000000
1,11153125,4/11/2017,Re-evaluation,3.0,2.0,NaN,NaN,NaN,4/1/2013,Active,...,No,No,NaN,NaN,NaN,4.166667,6.666667,NaN,NaN,NaN
2,11153125,10/11/2017,Re-evaluation,2.0,1.0,0.0,0.0,NaN,4/1/2013,Active,...,No,No,NaN,NaN,NaN,2.083333,4.444444,0.000000,0.000000,NaN
3,11153125,6/21/2018,Re-evaluation,19.0,19.0,4.0,8.0,0.0,4/1/2013,Active,...,No,No,NaN,NaN,NaN,39.583333,42.222222,19.047619,16.666667,0.000000
4,11153125,9/24/2018,Re-evaluation,10.0,11.0,3.0,6.0,2.0,4/1/2013,Active,...,No,No,NaN,NaN,NaN,22.916667,22.222222,14.285714,12.500000,16.666667


In [37]:
# map gender into Male, Female, and Other 
gender_mapping = {
    'Female': 'Female',
    'Male': 'Male',
    'Other:': 'Other',
    'Other: Born male, unsure of gender identity at this time': 'Other',
    'Other: Born male/ he/ him, Currently identifies as female/ she/ her': 'Other',
    'Other: Desires to transition from male to female': 'Other',
    'Other: Identifies as female': 'Other',
    'Other: Transgender female': 'Other',
    'Other: biologically female. self describes as male': 'Other',
    'Other: currently exploring male and female gender identities': 'Other',
    'Other: non binary': 'Other',
    'Other: not sure yet as to who Sam identifies with': 'Other'
}

# Apply all of this gender mapping to the Gender column
demographic_abcs['Gender'] = demographic_abcs['Gender'].map(gender_mapping).fillna('Other')

In [38]:
# create summary table of gender demographic 
unique_df = demographic_abcs.drop_duplicates(subset='Local ID')

gender_counts = unique_df['Gender'].value_counts().reset_index()
gender_counts.columns = ['Gender', 'Count']

gender_counts

,Gender,Count
0,Male,12973
1,Female,6433
2,Other,55


In [29]:
# clean race and map unknown/misidentified participants 
# Define a mapping dictionary for the new categories
race_mapping = {
    'White': 'White',
    'Black or African American': 'Black',
    'Unknown, not collected': 'Unknown',
    'Other: Hispanic': 'Hispanic',
    'Asian': 'Asian',
    'Other': 'Other',
    'Black or African American, White': 'Multiracial',
    'Other: Latino': 'Hispanic',
    'American Indian or Alaska Native': 'Native',
    'Other: hispanic': 'Hispanic',
    'Asian, White': 'Multiracial',
    'Other: Mexican': 'Hispanic',
    'Unknown, not collected, White': 'Unknown',
    'Native Hawaiian or Other Pacific Islander': 'Pacific Islander',
    'Other: Biracial': 'Multiracial',
    'Other: Hispanic, White': 'Multiracial',
    'Other: Bi-racial': 'Multiracial',
    'Other: East Indian': 'Asian',
    'Asian, Black or African American': 'Multiracial',
    'Other: Indian': 'Asian',
    'Black or African American, Unknown, not collected': 'Multiracial',
    'Other: Middle Eastern': 'Other',
    'American Indian or Alaska Native, White': 'Multiracial',
    'Other: Guyanese': 'Unknown',
    'Other: Filipino': 'Unknown',
    'Other: Hispanic - Puerto Rican descent': 'Hispanic',
    'Other: Puerto Rican': 'Hispanic',
    'Other: Mixed race': 'Multiracial',
    'Other: Mixed Race': 'Multiracial',
    'Native Hawaiian or Other Pacific Islander, White': 'Multiracial',
    'Other: "Hispanic"': 'Hispanic',
    'Other: Latina': 'Hispanic',
    'Other: Latin  American': 'Hispanic',
    'Other: Jordanian': 'Unknown',
    'Other: Pakistan, Unknown, not collected': 'Unknown',
    'Other: Hispanic/Latino': 'Hispanic',
    'Black or African American, Other: Hispanic': 'Multiracial',
    'Black or African American, Other: Guyanese': 'Multiracial',
    'Other: Russian': 'Unknown',
    'Other: West Indian': 'Unknown',
    'Asian, Other: Indian': 'Multiracial',
    'Other: HISPANIC': 'Hispanic',
    'Other: Guyanses': 'Unknown',
    'Asian, Other: Pakistani': 'Asian',
    'American Indian or Alaska Native, Black or African American': 'Multiracial',
    'Other: Ethiopian': 'Unknown',
    'Other: Egyptian': 'Unknown',
    'Other: biracial': 'Multiracial',
    'Other: Colombian': 'Unknown',
    'Other: Did not want to Identity with Race or Ethnicity': 'Unknown',
    'Other: Dominican': 'Unknown',
    'Other: Chinese': 'Unknown',
    'Other: Eastern European': 'Unknown',
    'Other: Ecuador': 'Unknown',
    'Other: Ecuadorian': 'Unknown',
    'Other: Caucasian and African Amercian': 'Multiracial',
    'Other: Caucasian': 'White',
    'Other: Eurasian': 'Multiracial',
    'Other: Carribean Indian': 'Other',
    'Other: Filipino and Caucasian': 'Multiracial',
    'Other: Bosnian': 'Other',
    'Other: Black Hispanic': 'Multiracial',
    'Other: HIspanic': 'Hispanic',
    'Other: Hispanic ': 'Hispanic',
    'Other: Hispanic (Uruguayan/Dominican)': 'Hispanic',
    'Other: Hispanic - Puerto Rican': 'Hispanic',
    'Other: Hispanic, AA and white': 'Multiracial',
    'Other: Hispanic, Unknown, not collected': 'Hispanic',
    'Other: Bi-Racial': 'Multiracial',
    'Other: Hispanic/ Latina': 'Hispanic',
    'Other: Bengalii': 'Other',
    'Other: Hispanic: Puerto Rican': 'Hispanic',
    'Other: Hondurian': 'Hispanic',
    'Other: Bengali': 'Other',
    'Other: Indian of Guyanese decent': 'Multiracial',
    'Other: Indian which decent unknown': 'Asian',
    'Other: Iraq': 'Unknown',
    'Other: Iraqi': 'Unknown',
    'Other: Irish': 'Unknown',
    'Other: Israel': 'Unknown',
    'Other: Jamaican American': 'Black',
    'Other: Jewish': 'Unknown',
    'Other: Bangladish': 'Unknown',
    'Other: Baltic ethnic group (Latvian)': 'Unknown',
    'Other: Arab': 'Other',
    'Other: Albian/European': 'Unknown',
    'Other: Latino ( Mexican & Puerto Rican )': 'Hispanic',
    'Other: Latino (Guatemalen & El Salvador)': 'Hispanic',
    'Other: Latino, White': 'Hispanic',
    'Other: MIXED RACE, White': 'Multiracial',
    'Other: MIddle Eastern- North African Descent, ': 'Other',
    'Other: Mauritius': 'Unknown',
    'Other: African-American/Vietnamese': 'Multiracial',
    'Other: Mexican/Puerto Rican': 'Unknown',
    'Other: Middle Eastern Decent': 'Other',
    'Other: South African': 'Unknown',
    'Other: South American': 'Unknown',
    'Other: Spanish': 'Unknown',
    'Other: Vietnamese': 'Unknown',
    'Other: Vietnamesse': 'Unknown',
    'Other: unknown': 'Unknown',
    'Other: White and Asian': 'Multiracial',
    'Other: White and Black or African America': 'Multiracial',
    'Other: puerto rican': 'Unknown',
    'Asian, Black or African American, White': 'Multiracial',
    'Asian, Native Hawaiian or Other Pacific Islander': 'Multiracial',
    'Asian, Other: Filipino': 'Multiracial',
    'Asian, Other: Hispanic': 'Multiracial',
    'Other: not checked off': 'Unknown',
    'Other: myanmar': 'Unknown',
    'Other: mother is white and father is 1/2 white': 'Multiracial',
    'Other: White, Hispanic': 'Multiracial',
    'Other: White/Indian': 'Multiracial',
    'Asian, Other: vietnamese': 'Multiracial',
    'Other: mixed race': 'Multiracial',
    'Other: hispanic, White': 'Multiracial',
    'Other: latino/white': 'Hispanic',
    'Other: mexican': 'Unknown',
    'American Indian or Alaska Native, Other: Native Hawaiian or Other Pacific Islander': 'Multiracial',
    'Other:': 'Other'  
}

# Group by "Type of Service" and "Race" and calculate mean ABC scores
demographic_abcs['Race'] = demographic_abcs['Race'].map(race_mapping).fillna('Other')


In [30]:
# create summary table of gender demographic 
unique_df = demographic_abcs.drop_duplicates(subset='Local ID')
race_counts = unique_df['Race'].value_counts().reset_index()
race_counts.columns = ['Race', 'Count']


,Race,Count
0,White,11969
1,Black,4146
2,Hispanic,998
3,Unknown,895
4,Asian,585
5,Multiracial,430
6,Other,309
7,Native,106
8,Pacific Islander,23


In [11]:
# save all resulting data frames to data file
demographic_abcs.to_csv('../data/demographic_abcs.csv', index=False)  
gender_counts.to_csv('../data/gender_counts.csv', index=False)
race_counts.to_csv('../data/race_counts.csv', index=False)